In [1]:
import boto3
import json
from botocore.config import Config
import tqdm

In [2]:
config = Config(read_timeout=200)
session = boto3.Session(profile_name="default")
client = boto3.client("bedrock-agent-runtime", region_name="us-west-2", config=config)
FLOW_IDENTIFIER = ""

In [ ]:
def invoke_flow(flowAliasIdentifier, context):
    try:
        response = client.invoke_flow(
            flowIdentifier=FLOW_IDENTIFIER,
            flowAliasIdentifier=flowAliasIdentifier, 
            inputs=[
                {
                    "content": {"document": context},
                    "nodeName": "FlowInputNode",
                    "nodeOutputName": "document"
                }
            ]
        )
        for event in response.get('responseStream'):
            if 'flowOutputEvent' in event:
                if event['flowOutputEvent']['nodeName'] == 'Flow':
                    return event['flowOutputEvent']['content']['document']
        else:
            return False
        
    except Exception as e:
        print(f"An error occured when invoking flow with Alias ID {flowAliasIdentifier}: {e}")
        return False

## Run debate

In [ ]:
test_questions_file = "test.json"

with open(test_questions_file, 'r') as f:
    data = json.load(f)

for entry in tqdm.tqdm(data.values(), desc="Processing interventions"):

    intervention = entry["intervention"]
    intervention_id = entry["intervention_id"]
    questions = entry["cqs"]

    # Prepare debate
    debate = {}
    output_file = f"debates/debate_{intervention_id}.json"

    # Load debate data
    questions = '\n'.join([f"{i+1}. {q["cq"]}" for i, q in enumerate(questions)])
    
    debate["intervention"] = intervention
    debate["questions"] = questions

    # Opening statements
    opening_statement_1 = invoke_flow('JFFQRXHJY0', str(debate))
    debate["opening_statement_1"] = opening_statement_1
    opening_statement_2 = invoke_flow('JFFQRXHJY0', str(debate))
    debate["opening_statement_2"] = opening_statement_2

    # Rebuttals
    rebuttal_1 = invoke_flow('3LGFTPJQF3', str(debate))
    debate["rebuttal_1"] = rebuttal_1
    rebuttal_2 = invoke_flow('3LGFTPJQF3', str(debate))
    debate["rebuttal_2"] = rebuttal_2

    # Closing statements
    closing_statement_1 = invoke_flow('QAQCBAD9NE', str(debate))
    debate["closing_statement_1"] = closing_statement_1
    closing_statement_2 = invoke_flow('QAQCBAD9NE', str(debate))
    debate["closing_statement_2"] = closing_statement_2

    # Judge's decision
    verdict = invoke_flow('E57ITLRQST', (str(debate)))
    debate["verdict"] = verdict

    # Save debate to JSON
    with open(output_file, "w") as f:
        json.dump(debate, f, indent=4)

    print(f"Saved debate to: {output_file}")